# Exercise 1a & 1b — Transformer Summarization & Question Answering
**Paper:** Vaswani et al., *"Attention Is All You Need"* (2017)  
**Focus Sections:** Positional Encoding · Why Self-Attention

---

## Overview

This notebook explores the use of pre-trained Transformer models from the HuggingFace library for two NLP tasks:

1. **Text Summarization** summarizing key sections of the original Transformers paper and comparing results across models, parameters, and techniques.
2. **Question Answering (QA)** extracting specific answers from the same paper sections using extractive QA models.

---

## Setup

Installing a pinned version of `transformers==4.35.0` to ensure compatibility with all models used in this notebook.

In [ ]:
!pip install transformers==4.35.0

## 1. Input Text Preparation

The two sections assigned to our group are manually extracted from the paper and stored as plain text variables.

**Preprocessing decisions:**
- Mathematical notation (e.g. `π`) was replaced with readable alternatives (`pi`) to reduce tokenizer noise without losing meaning.
- Citation numbers and reference markers were removed as they add no value to the summarization.
- Each section is stored separately to allow both isolated and combined summarization.

In [ ]:
positional_encoding_text = '''Since our model contains no recurrence and no convolution, in order for the model to make use of the order of the sequence, we must inject some information about the relative or absolute position of the tokens in the sequence. To this end, we add "positional encodings" to the input embeddings at the bottoms of the encoder and decoder stacks. The positional encodings have the same dimension dmodel
as the embeddings, so that the two can be summed. There are many choices of positional encodings, learned and fixed. In this work, we use sine and cosine functions of different frequencies
where pos is the position and i is the dimension. That is, each dimension of the positional encoding
corresponds to a sinusoid. The wavelengths form a geometric progression from 2 pi to 10000 · 2 pi. We
chose this function because we hypothesized it would allow the model to easily learn to attend by
relative positions, since for any fixed offset k, P Epos+k can be represented as a linear function of
P Epos.
We also experimented with using learned positional embeddings instead, and found that the two
versions produced nearly identical results. We chose the sinusoidal version
because it may allow the model to extrapolate to sequence lengths longer than the ones encountered
during training.
'''

why_self_attention_text = '''In this section we compare various aspects of self-attention layers to the recurrent and convolutional layers commonly used for mapping one variable-length sequence of symbol representations
(x1, ..., xn) to another sequence of equal length (z1, ..., zn), such as a hidden
layer in a typical sequence transduction encoder or decoder. Motivating our use of self-attention we
consider three desiderata.
One is the total computational complexity per layer. Another is the amount of computation that can
be parallelized, as measured by the minimum number of sequential operations required.
The third is the path length between long-range dependencies in the network. Learning long-range
dependencies is a key challenge in many sequence transduction tasks. One key factor affecting the
ability to learn such dependencies is the length of the paths forward and backward signals have to
traverse in the network. The shorter these paths between any combination of positions in the input
and output sequences, the easier it is to learn long-range dependencies. Hence we also compare
the maximum path length between any two input and output positions in networks composed of the
different layer types.
As noted in Table 1, a self-attention layer connects all positions with a constant number of sequentially
executed operations, whereas a recurrent layer requires O(n) sequential operations. In terms of
computational complexity, self-attention layers are faster than recurrent layers when the sequence
length n is smaller than the representation dimensionality d, which is most often the case with
sentence representations used by state-of-the-art models in machine translations, such as word-piece
 and byte-pair  representations. To improve computational performance for tasks involving
very long sequences, self-attention could be restricted to considering only a neighborhood of size r in
the input sequence centered around the respective output position. This would increase the maximum
path length to O(n/r). We plan to investigate this approach further in future work.
A single convolutional layer with kernel width k < n does not connect all pairs of input and output
positions. Doing so requires a stack of O(n/k) convolutional layers in the case of contiguous kernels,
or O(logk(n)) in the case of dilated convolutions, increasing the length of the longest paths
between any two positions in the network. Convolutional layers are generally more expensive than
recurrent layers, by a factor of k. Separable convolutions, however, decrease the complexity
considerably, to O(k · n · d + n · d). Even with k = n, however, the complexity of a separable
convolution is equal to the combination of a self-attention layer and a point-wise feed-forward layer,
the approach we take in our model.
As side benefit, self-attention could yield more interpretable models. We inspect attention distributions
from our models and present and discuss examples in the appendix. Not only do individual attention
heads clearly learn to perform different tasks, many appear to exhibit behavior related to the syntactic
and semantic structure of the sentences.'''

## 2. Environment Setup

Loading PyTorch and the HuggingFace `pipeline` API.

In [ ]:
import torch
from transformers import pipeline
import pandas as pd

device = torch.device("cuda:0" if torch.cuda.is_available() else "cpu")
print(f"Using device: {device}")

/usr/local/lib/python3.12/dist-packages/transformers/utils/generic.py:441: FutureWarning: `torch.utils._pytree._register_pytree_node` is deprecated. Please use `torch.utils._pytree.register_pytree_node` instead.
  _torch_pytree._register_pytree_node(
/usr/local/lib/python3.12/dist-packages/transformers/utils/generic.py:309: FutureWarning: `torch.utils._pytree._register_pytree_node` is deprecated. Please use `torch.utils._pytree.register_pytree_node` instead.
  _torch_pytree._register_pytree_node(


Using device: cuda:0


## 3. Text Summarization — Model & Parameter Comparison

### Models
Two summarization models were selected for comparison:

| Model | Architecture | Trained On | Link |
|---|---|---|---|
| `facebook/bart-large-cnn` | BART-large | CNN/DailyMail news | https://huggingface.co/facebook/bart-large-cnn |
| `Falconsai/text_summarization` | Fine-tuned T5-style | Mixed | https://huggingface.co/Falconsai/text_summarization |


### Parameters
- **`max_length=100`** for individual sections, **`max_length=150`** for combined text.
- **`num_beams=[1, 4]`** — beam search depth is varied to observe its effect on summary quality. Greedy decoding (`beams=1`) takes the locally best token at each step, while `beams=4` explores multiple paths and selects the globally most probable sequence.
- **`min_length`** — not set explicitly; for short academic sections the model output is already sufficiently long without a lower bound.
- **`length_penalty`** — not applied; most impactful on longer texts where output length variance is higher.

### Context Strategies
Three summarization contexts are compared:
- **Separate**: each section summarized independently for maximum clarity per topic.
- **Combined**: both sections concatenated with labeled headers to test cross-section summarization.
- **Chunked + Iterative**: combined text split into overlapping chunks, each summarized individually, then joined and re-summarized for a final condensed output.

In [ ]:
models = ['facebook/bart-large-cnn', 'Falconsai/text_summarization']
num_beams = [1, 4]
summary_dic = {}

combined_text = "Positional Encoding: " + positional_encoding_text + "\n\nWhy Self-Attention: " + why_self_attention_text

### Chunking Helper Function

For longer inputs, a sliding window chunking strategy is used to stay within model token limits.

- `chunk_size=200:` words per chunk
- `overlap=50:` the last 50 words of each chunk are repeated at the start of the next chunk. This overlap preserves context continuity across chunk boundaries.

In [ ]:
def chunk_text(text: str, chunk_size: int = 200, overlap: int = 50) -> list[str]:
    """
    Splits a given text into overlapping chunks of words.
    """
    words = text.split()
    chunks = []
    start = 0
    while start < len(words):
        chunk = words[start : start + chunk_size]
        chunks.append(' '.join(chunk))
        start += chunk_size - overlap
    return chunks

### Summarization Loop

For each combination of model and beam count:
1. Summarize the Positional Encoding section independently.
2. Summarize the Why Self-Attention section independently.
3. Summarize the combined labeled text.
4. Apply chunking to the combined text, summarize each chunk (`max_length=60`), join the chunk summaries, then apply a final iterative summarization pass (`max_length=150`) if the joined text exceeds 50 words, otherwise return the joined summaries directly.

In [ ]:
for model in models:
  summarizer = pipeline("summarization", model=model, device=device)

  for n in num_beams:
    summary_pe = summarizer(positional_encoding_text, max_length=100, do_sample=False, num_beams=n)
    summary_sa = summarizer(why_self_attention_text, max_length=100, do_sample=False, num_beams=n)
    summary_combined = summarizer(combined_text, max_length=150, do_sample=False, num_beams=n)

    chunks = chunk_text(combined_text)
    chunk_summaries = [summarizer(chunk, max_length=60)[0]['summary_text']
                   for chunk in chunks]
    joined = ' '.join(chunk_summaries)
    if len(joined.split()) > 50:
      final_summary = summarizer(joined, max_length=150, do_sample=False)[0]['summary_text']
    else:
      final_summary = joined

    summary_dic[(model, n)] = [
        summary_pe[0]['summary_text'],
        summary_sa[0]['summary_text'],
        summary_combined[0]['summary_text'],
        final_summary,
        n
    ]

/usr/local/lib/python3.12/dist-packages/transformers/generation/configuration_utils.py:418: UserWarning: `num_beams` is set to 1. However, `early_stopping` is set to `True` -- this flag is only used in beam-based generation modes. You should set `num_beams>1` or unset `early_stopping`.
  warnings.warn(
/usr/local/lib/python3.12/dist-packages/transformers/generation/configuration_utils.py:437: UserWarning: `num_beams` is set to 1. However, `length_penalty` is set to `2.0` -- this flag is only used in beam-based generation modes. You should set `num_beams>1` or unset `length_penalty`.
  warnings.warn(
/usr/local/lib/python3.12/dist-packages/transformers/pipelines/base.py:1101: UserWarning: You seem to be using the pipelines sequentially on GPU. In order to maximize efficiency please use a dataset
  warnings.warn(
Token indices sequence length is longer than the specified maximum sequence length for this model (693 > 512). Running this sequence through the model will result in indexing er

### Results & Observations

**Key findings:**

- **`Falconsai` with `beams=1`** produced the best overall summaries, capturing the sinusoidal extrapolation rationale, the three desiderata mention, and computational complexity comparisons.
- **`beams=4` did not consistently improve quality.** More beam paths led to more conservative, generic outputs, likely because the model defaults to statistically safe phrasing.
- **Combined context summaries** tended to be dominated by the Positional Encoding section, with Self-Attention content underrepresented.
- **Chunking + iterative summarization** produced the most comprehensive Final Summary, covering both sections more evenly than direct combined summarization.
- **Domain mismatch** is the primary limitation: all models were trained on news data.

In [ ]:
data = []
for (model, beams), summaries in summary_dic.items():
    display_final_summary = summaries[3] if summaries[4] == 1 else ""
    display_final_len = len(summaries[3]) if summaries[4] == 1 else 0

    data.append({
        'Model': model,
        'Beams': summaries[4],
        'Positional Encoding Summary': summaries[0],
        'Self-Attention Summary': summaries[1],
        'Combined Summary': summaries[2],
        'Final Summary': display_final_summary,
        'PE_Len': len(summaries[0]),
        'SA_Len': len(summaries[1]),
        'Comb_Len': len(summaries[2]),
        'Final_Len': display_final_len
    })

df_results = pd.DataFrame(data)

print("Comparison of Summaries (Final Summary only for Beams 1):")
display(df_results[['Model', 'Beams', 'Positional Encoding Summary', 'Self-Attention Summary', 'Combined Summary', 'Final Summary']])

Comparison of Summaries (Final Summary only for Beams 1):


,Model,Beams,Positional Encoding Summary,Self-Attention Summary,Combined Summary,Final Summary
0,facebook/bart-large-cnn,1,In order for the model to make use of the orde...,Self-attention layers are used to map sequence...,In order to make use of the order of the seque...,Since our model contains no recurrence and no ...
1,facebook/bart-large-cnn,4,Since our model contains no recurrence and no ...,Self-attention layers are faster than recurren...,In order for the model to make use of the orde...,
2,Falconsai/text_summarization,1,We use sine and cosine functions of different ...,In this section we compare various aspects of ...,We use sine and cosine functions of different ...,Since our model contains no recurrence and no ...
3,Falconsai/text_summarization,4,In order for the model to make use of the orde...,In this section we compare various aspects of ...,"In this work, we use sine and cosine functions...",


---

## 4. Question Answering

### Questions
Questions were designed to target key concepts from each section that the summarization models frequently missed:

**Positional Encoding:**
- *"Which version allows the model to extrapolate to sequence lengths longer than the ones encountered during training?"*
- *"Because the model does not contain convolution, what needs to be done so the model can use the order of the sequence?"*

**Why Self-Attention:**
- *"How is the amount of parallel computation measured?"*
- *"What is a side benefit of self-attention?"*

In [ ]:
questions = {"pos_emb": [
                        "Which version allows the modelto extrapolate to sequence lengths longer than the ones encountered during training?",
                        "Because the model does not contain convolution what need to be done, so the model can use the order of sequence?"
              ],
             "self_att": [
                 "How is the amout of parallel computation measured?",
                 "What is a side benefit of seld-attention?"
             ]}

### QA Models

| Model | Architecture | Fine-tuned On | Link |
|---|---|---|---|
| `google-bert/bert-large-uncased-whole-word-masking-finetuned-squad` | BERT-large | SQuAD | https://huggingface.co/google-bert/bert-large-uncased-whole-word-masking-finetuned-squad |
| `Falconsai/question_answering_v2` | DistilBERT | QA dataset | https://huggingface.co/Falconsai/question_answering_v2 |

In [ ]:
qa_models = ["google-bert/bert-large-uncased-whole-word-masking-finetuned-squad", "Falconsai/question_answering_v2"]
qa_dic = {}

### QA Loop

For each model, questions are run against three context types:
1. **Isolated section context:** question matched to its relevant section only.
2. **Combined context*:* all questions run against the full concatenated text.

Both the extracted answer and the model's **confidence score** are recorded. The score reflects how certain the model is that the extracted span answers the question.

In [ ]:
for model in qa_models:
  qa = pipeline("question-answering", model=model, handle_impossible_answer=True, device=device)

  for question in questions["pos_emb"]:
    answer = qa(question=question, context=positional_encoding_text)
    qa_dic[
        (model, question, "pos_emb_context")
    ] = {'answer': answer['answer'], 'score': answer['score']}

  for question in questions["self_att"]:
    answer = qa(question=question, context=why_self_attention_text)
    qa_dic[
        (model, question, "self_att_context")
    ] = {'answer': answer['answer'], 'score': answer['score']}

  all_questions_for_combined_context = questions['pos_emb'] + questions['self_att']
  for question in all_questions_for_combined_context:
      answer = qa(question=question, context=combined_text)
      qa_dic[
          (model, question, "combined_text_context")
      ] = {'answer': answer['answer'], 'score': answer['score']}

/usr/local/lib/python3.12/dist-packages/transformers/utils/generic.py:309: FutureWarning: `torch.utils._pytree._register_pytree_node` is deprecated. Please use `torch.utils._pytree.register_pytree_node` instead.
  _torch_pytree._register_pytree_node(
Some weights of the model checkpoint at google-bert/bert-large-uncased-whole-word-masking-finetuned-squad were not used when initializing BertForQuestionAnswering: ['bert.pooler.dense.weight', 'bert.pooler.dense.bias']
- This IS expected if you are initializing BertForQuestionAnswering from the checkpoint of a model trained on another task or with another architecture (e.g. initializing a BertForSequenceClassification model from a BertForPreTraining model).
- This IS NOT expected if you are initializing BertForQuestionAnswering from the checkpoint of a model that you expect to be exactly identical (initializing a BertForSequenceClassification model from a BertForSequenceClassification model).


### QA Results & Observations

**Key findings:**

- **`Falconsai` achieved higher confidence scores** across almost all questions. However, higher confidence does not always mean a more accurate answer.
- **Q2 had low confidence for both models** (BERT: 0.07, Falconsai: 0.18). This suggests the question phrasing was ambiguous relative to the context, the model found it harder to identify the exact answer span.
- **BERT was more accurate but less confident on Q2:** it extracted the correct action despite low confidence, while Falconsai extracted the goal rather than the action.
- **Combined context degraded QA quality significantly.** BERT answered Q2 with *"a stack of O(n/k) convolutional layers"* and Falconsai with *"dilated"* — both wrong. More context caused the models to lose focus and retrieve answers from the wrong section.

---

## 5. Conclusions & Limitations

### Summary of Findings

| Aspect | Finding |
|---|---|
| Best summarization model | `Falconsai/text_summarization` with `num_beams=1` |
| Best summarization technique | Chunking + iterative summarization |
| Best QA model | `Falconsai/question_answering_v2`  |
| Effect of `num_beams` | `beams=1` outperformed `beams=4` for short academic text |
| Combined vs. isolated context | Isolated context produced better results for both summarization and QA |

### Limitations

- **Domain mismatch**: All models were pre-trained on news data. Academic writing structures arguments differently, causing models to over-weight introductory sentences and miss key reasoning mid-paragraph.
- **Summaries partially align with the paper**: Core concepts are captured but nuanced reasoning is consistently missed.
- **Confidence ≠ correctness**: High QA confidence scores do not guarantee accurate answers.

In [ ]:
qa_data = []
for (model, question, context_type), result in qa_dic.items():
    qa_data.append({
        'Model': model,
        'Question': question,
        'Context Type': context_type,
        'Answer': result['answer'],
        'Score': result['score']
    })

df_qa_results = pd.DataFrame(qa_data)
display(df_qa_results)

,Model,Question,Context Type,Answer,Score
0,google-bert/bert-large-uncased-whole-word-mask...,Which version allows the modelto extrapolate t...,pos_emb_context,sinusoidal,0.586703
1,google-bert/bert-large-uncased-whole-word-mask...,Because the model does not contain convolution...,pos_emb_context,inject some information about the relative or ...,0.073810
2,google-bert/bert-large-uncased-whole-word-mask...,How is the amout of parallel computation measu...,self_att_context,minimum number of sequential operations required,0.526031
3,google-bert/bert-large-uncased-whole-word-mask...,What is a side benefit of seld-attention?,self_att_context,more interpretable models,0.713349
4,google-bert/bert-large-uncased-whole-word-mask...,Which version allows the modelto extrapolate t...,combined_text_context,sinusoidal,0.567536
5,google-bert/bert-large-uncased-whole-word-mask...,Because the model does not contain convolution...,combined_text_context,a stack of O(n/k) convolutional layers,0.287500
6,google-bert/bert-large-uncased-whole-word-mask...,How is the amout of parallel computation measu...,combined_text_context,minimum number of sequential operations required,0.543061
7,google-bert/bert-large-uncased-whole-word-mask...,What is a side benefit of seld-attention?,combined_text_context,more interpretable models,0.484603
8,Falconsai/question_answering_v2,Which version allows the modelto extrapolate t...,pos_emb_context,sinusoidal version,0.994831
9,Falconsai/question_answering_v2,Because the model does not contain convolution...,pos_emb_context,make use of the order of the sequence,0.178815
